# EGO Step2 F0 — Colab A100 GPU 스모크

F0 (WM-only GRPO) 학습 코드가 이 GPU 에서 **처음부터 끝까지 도는지**를 검증한다.
학습 품질/수렴은 검증 대상이 아니다 (그건 실데이터 + full run 의 몫).

런타임: **A100 40GB** 권장 (런타임 → 런타임 유형 변경 → GPU: A100).

두 모드:
- `--wm synth` (기본): 다운로드 없이 스키마-정합 합성 jsonl 로 Qwen3-VL GRPO 루프만 검증. 수 분.
- `--wm real`: 공식 V-JEPA2 ViT-g/384 백본 + EK100 AC probe 를 실제로 받아 GPU 로드·forward → 실 후보 산출 → GRPO 루프. 다운로드 ~수 GB (백본이 큼).

In [ ]:
# 1) GPU 확인
!nvidia-smi -L

In [ ]:
# 2) 리포 준비 (push 된 브랜치를 clone). 새 러너 파일이 커밋돼 있어야 한다.
%cd /content
!git clone https://github.com/hublemon/EGO.git 2>/dev/null || (cd EGO && git pull)
%cd /content/EGO

## 모드 A — 합성 데이터 스모크 (기본, 다운로드 없음)

모델 로드 → 생성 → WM-likelihood reward → LoRA 업데이트 → 체크포인트까지 검증.

In [ ]:
!python scripts/step2/colab_smoke_f0.py --wm synth

## 모드 B — 공식 V-JEPA2 실추론 스모크 (선택)

백본/probe 를 매 세션 다시 받지 않으려면 `--assets_dir` 를 Google Drive 로 지정한다.
아래 mount 셀은 선택 — Drive 캐시를 쓸 때만 실행.

In [ ]:
# (선택) Drive 마운트 — 체크포인트 캐시 재사용
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python scripts/step2/colab_smoke_f0.py --wm real \
    --assets_dir /content/drive/MyDrive/ego_vjepa_assets

## 참고

- OOM 이면 더 작은 정책모델로: `--model Qwen/Qwen2.5-VL-7B-Instruct`
- 실데이터 jsonl 이 있으면: `--train_jsonl /content/drive/MyDrive/.../grpo_train.jsonl`
- 규모 조절 플래그: `--n_samples --max_steps --num_generations --per_device_batch --max_completion_length`
- 이미 설치된 환경이면: `--no_install`